# **Lab 2 : Direct method of Interpolation**<br>

Interpolation is the process of estimating unknown data that lies within the range of some known data. The simplest interpolation is the **Direct method of Interpolation**. In order to interpolate unknown data, one needs to define a model that would fit the known data and based on that model, one would obtain the unknown data. This model would basically be a function that tries to satisfy the known data points. There can be various types of functions that may satisfy the same given data points, but for interpolation, we usually choose **polynomials** as interpolating functions due to the fact that polynomials are very easy to evaluate, differentiate and integrate.

In today's lab, we will be implementing the Direct method of Interpolation in python. To test our implementation, we will be using the same data that we used during our class lectures. Run the following cell to load the known data points:


In [ ]:
t = [0, 10, 15, 20, 22.5, 30]
v = [0, 227.04, 362.78, 517.35, 602.97, 901.67]

##Task 1
Design a utility function that will be called in the `DirectInterpolation` function. The purpose of this function will be to find the $n+1$ closest points to the unknown value $t_{new}$ where we want to interpolate the data, where $n$ is the order of the interpolating polynomial. Understand that the nearest points should be selected such that they bracket the $t_{new}$. The function to be implemented is as follows:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def NearestPoints(t, v, n, t_new):

    left_points = []
    right_points = []

    for i in range(len(t)):

        distance = t[i] - t_new

        if distance < 0:
            left_points.append([abs(distance), t[i], v[i]])

        elif distance > 0:
            right_points.append([abs(distance), t[i], v[i]])

        else:
            left_points.append([0, t[i], v[i]])

    left_points.sort(key=lambda x: x[0])
    right_points.sort(key=lambda x: x[0])

    selected = []

    l = 0
    r = 0

    while len(selected) < n:

        if r < len(right_points):
            selected.append(right_points[r])
            r += 1

            if len(selected) == n:
                break

        if l < len(left_points):
            selected.append(left_points[l])
            l += 1


    selected.sort(key=lambda x: x[1])

    t_nearest = [p[1] for p in selected]
    v_nearest = [p[2] for p in selected]

    return t_nearest, v_nearest


The above function is supposed to return two vectors, say `t_nearest` and `v_nearest` consisting of $n+1$ elements each. For testing how the function works, we can print the data and see what we are getting. If our implementation is correct, then we should be getting $[15], [362.78]$ for $n=0$. Run the following cell to check this:

In [ ]:
n = 0
t_new = 16

t_nearest, v_nearest = NearestPoints(t, v, n, t_new)
print(t_nearest)
print(v_nearest)

[]
[]


In [ ]:
n = 1
t_new = 16

t_nearest, v_nearest = NearestPoints(t, v, n, t_new)
print(t_nearest)
print(v_nearest)

[20]
[517.35]


The output for $n=1$ should be $[15, 20], [362.78, 517.35]$

In [ ]:
n = 2
t_new = 16

t_nearest, v_nearest = NearestPoints(t, v, n, t_new)
print(t_nearest)
print(v_nearest)

[15, 20]
[362.78, 517.35]


The output for $n=2$ should be $[15, 20, 10], [362.78, 517.35, 227.04]$

## Task 2
Now your task is to design a generalized function that takes the given data as parameters as well as the order $n$ of the polynomial and uses the *Direct Interpolation* method to interpolate the unknown data at $t_{new}$. The following cell contains the function to be implemented:

In [ ]:
def DirectInterpolation(t, v, n, t_new):
    t_nearest, v_nearest = NearestPoints(t, v, n, t_new)
    f = []
    X = np.zeros((n+1, n+1))

    for i in range(n+1):
        for j in range(n+1):
            X[i][j] = t_nearest[i]**j

    Y = np.zeros((n+1, 1))
    for i in range(n+1):
        Y[i][0] = v_nearest[i]

    A = np.linalg.inv(X)@(Y)

    f.append(A[0][0])
    for i in range(1, n+1):
        f.append(A[i][0])
    return f

def eval(f, t_new):
    val = 0
    for i in range(len(f)):
        val += f[i]*(t_new**i)
    return val

Note that the above function will at first need to find a number of closest points to the unknown data $t_{new}$, and this number will vary depending on the order $n$ of the polynomial that we want to use as the interpolating function. So you will have to call the function you implemented in Task 1 inside this function.

In [ ]:
print(eval(DirectInterpolation(t, v, 1, 16),16))

IndexError: list index out of range

For $n=1$, we should get $v(16) = 393.7ms^{-1}$.

In [ ]:
print(eval(DirectInterpolation(t, v, 2, 16),16))

For $n=2$, we should get $v(16) = 392.19ms^{-1}$.

In [ ]:
print(eval(DirectInterpolation(t, v, 3, 16),16))

For $n=3$, we should get $v(16) = 392.06ms^{-1}$.

##Task 3
Now, test the function using different order of polynomials, setting $n = 1, 2,$ and $3$ and so on and print the absolute relative approximate error at each step. Also generate a plot of Order $(n)$ vs Relative Approximate Error $(|\epsilon_a|\%)$. In case you do not remember how graphs can be plotted in Python, here is a sample code showing how to do it using the `matplotlib` library.

In [ ]:
errors = []
errors.append(0)
iters = []
prev = 0

for i in range(5):
  n = i+1
  f = DirectInterpolation(t, v, n, 16)
  current = eval(f, 16)
  iters.append(current)
  if n > 1:
    error = (abs((iters[i]-iters[prev] )/iters[prev]))*100
    errors.append(error)
    prev+=1
    print("Error at iteration " + str(i) + " : " + str(error))





In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vals = [1,2,3,4,5]

plt.plot(vals, errors, marker='o')
plt.title("Order(n) vs Relatve approx error")
plt.xlabel("Order(n)")
plt.ylabel("Errors")
plt.show()

##Task 4
Now, write a function for evaluating the acceleration at $t = 16$

In [ ]:
def acceleration(f, t_new):
  a = 0
  for i in range(1, len(f)):
    a += f[i]*(t_new**(i-1))*(i)
  return a

In [ ]:
print(acceleration(DirectInterpolation(t, v, 1, 16),16))

In [ ]:
print(acceleration(DirectInterpolation(t, v, 2, 16),16))

In [ ]:
print(acceleration(DirectInterpolation(t, v, 3, 16),16))